# 05 - Transformacion: Bronce a Plata
## Limpieza, Estandarizacion y Enriquecimiento de Datos

---

### Objetivo de este notebook

Este notebook implementa la transformacion de la **capa Bronce** a la **capa Plata**,
donde se aplican las transformaciones de **negocio** que convierten datos tecnicamente
correctos (pero crudos) en datos listos para analisis.

### Transformaciones aplicadas

| Paso | Transformacion | Proposito |
|------|---------------|----------|
| 1 | Renombramiento de columnas | Estandarizar nombres a espanol descriptivo |
| 2 | Filtrado de registros invalidos | Eliminar registros sin resultado de actividad |
| 3 | Conversion de tipos de datos | Convertir Activity Value a tipo double para calculos |
| 4 | Eliminacion de duplicados | Garantizar unicidad de registros |
| 5 | Enriquecimiento con propiedades | Agregar peso molecular, XLogP y complejidad |
| 6 | Clasificacion Lipinski | Evaluar biodisponibilidad oral segun la Regla de los 5 |

### Tabla de salida

- `pubchem_plata.actividades_biologicas` - Tabla consolidada con datos limpios y enriquecidos

### Regla de los 5 de Lipinski

La **Regla de Lipinski** (tambien conocida como Rule of Five) es un conjunto de criterios
empiricos que predicen si un compuesto tiene probabilidad de ser **biodisponible por via oral**:

| Criterio | Umbral | Justificacion |
|----------|--------|---------------|
| Peso molecular | <= 500 Da | Moleculas grandes no se absorben bien |
| XLogP (lipofilicidad) | <= 5 | Compuestos muy lipofílicos no se disuelven |
| Donantes de H | <= 5 | Demasiados donantes impiden cruzar membranas |
| Aceptores de H | <= 10 | Demasiados aceptores impiden cruzar membranas |

---
## 1. Importacion de Librerias y Configuracion

In [ ]:
from pyspark.sql import functions as F      # Funciones de transformacion de Spark SQL
from pyspark.sql.types import DoubleType     # Tipo de dato double para conversiones numericas

In [ ]:
# Configuracion de catalogo y esquemas
CATALOGO = spark.sql("SELECT current_catalog()").collect()[0][0]
ESQUEMA_BRONCE = "pubchem_bronce"  # Esquema de origen (datos crudos en Delta)
ESQUEMA_PLATA = "pubchem_plata"    # Esquema de destino (datos limpios y estandarizados)

print(f"Catalogo: {CATALOGO}")
print(f"Origen:   {ESQUEMA_BRONCE}")
print(f"Destino:  {ESQUEMA_PLATA}")

---
## 2. Carga de Tablas Fuente

Cargamos las dos tablas de la capa Bronce que se usaran como fuente:
- **bioactividad**: Resultados de bioensayos (tabla principal a transformar)
- **propiedades_compuestos**: Propiedades fisicoquimicas (para enriquecimiento)

In [ ]:
# Cargar tabla principal de bioactividad desde la capa Bronce
df_bioactividad = spark.table(f"{ESQUEMA_BRONCE}.bioactividad")

# Cargar tabla de propiedades para el enriquecimiento posterior
df_compuestos = spark.table(f"{ESQUEMA_BRONCE}.propiedades_compuestos")

# Registrar total de registros de origen para calcular tasa de retencion
total_bronce = df_bioactividad.count()

print(f"Registros en bioactividad (Bronce): {total_bronce:,}")
print(f"Compuestos en propiedades:          {df_compuestos.count()}")

---
## 3. Paso 1: Renombrar Columnas a Espanol Estandarizado

Se renombran todas las columnas de la tabla de bioactividad del formato
snake_case en ingles (heredado de PubChem) a nombres descriptivos en espanol.

Se utiliza un **diccionario de mapeo explicito** para garantizar trazabilidad
y documentar el origen de cada columna.

Tambien se eliminan las columnas de trazabilidad de la capa Bronce
(prefijo `_bronce_`) ya que la capa Plata tendra su propia trazabilidad.

In [ ]:
# Diccionario de mapeo: nombre en Bronce (ingles) -> nombre en Plata (espanol)
# Cada entrada documenta el origen y destino de la columna
MAPEO_COLUMNAS = {
    "aid": "id_ensayo",                    # Assay ID -> Identificador del ensayo
    "panel_member_id": "id_panel",          # Panel Member ID -> Identificador del panel
    "sid": "id_sustancia",                  # Substance ID -> Identificador de la sustancia
    "cid": "id_compuesto",                  # Compound ID -> Identificador del compuesto
    "activity_outcome": "resultado_actividad",  # Resultado del bioensayo
    "target_accession": "acceso_objetivo",  # Accession del objetivo biologico
    "target_gene_id": "id_gen_objetivo",    # Gene ID del objetivo biologico
    "activity_value_um": "valor_actividad_um",  # Valor de actividad en micromolar
    "activity_name": "nombre_actividad",    # Nombre de la actividad medida
    "assay_name": "nombre_ensayo",          # Nombre del ensayo biologico
    "assay_type": "tipo_ensayo",            # Tipo de ensayo (Screening, Confirmatory, etc.)
    "pubmed_id": "id_pubmed",              # ID de la publicacion en PubMed
    "rnai": "arni",                        # RNA de interferencia
    "nombre_comun": "nombre_comun"          # Nombre comun del medicamento (sin cambio)
}

# Aplicar renombramiento iterando sobre el diccionario de mapeo
df_renombrado = df_bioactividad
for columna_original, columna_nueva in MAPEO_COLUMNAS.items():
    if columna_original in df_bioactividad.columns:
        df_renombrado = df_renombrado.withColumnRenamed(columna_original, columna_nueva)

# Eliminar columnas de trazabilidad de Bronce (prefijo _bronce_)
# La capa Plata agregara su propia trazabilidad
columnas_finales = [c for c in df_renombrado.columns if not c.startswith("_")]
df_renombrado = df_renombrado.select(columnas_finales)

# Mostrar las columnas resultantes
print("Columnas renombradas a espanol:")
for columna in df_renombrado.columns:
    print(f"  - {columna}")

---
## 4. Paso 2: Filtrar Registros sin Resultado de Actividad

Los registros donde `resultado_actividad` es nulo o vacio no aportan
informacion util para el analisis de bioactividad y se descartan.

Estos registros pueden originarse cuando el API de PubChem retorna
filas incompletas por ensayos aun en proceso o con errores.

In [ ]:
# Contar registros antes del filtro para medir el impacto
antes_filtro = df_renombrado.count()

# Filtrar: mantener solo registros con resultado de actividad valido
df_filtrado = df_renombrado.filter(
    (F.col("resultado_actividad").isNotNull()) &  # No nulo
    (F.col("resultado_actividad") != "")           # No cadena vacia
)

# Contar registros despues del filtro
despues_filtro = df_filtrado.count()
eliminados = antes_filtro - despues_filtro

# Reportar impacto del filtro
print(f"Registros antes del filtro:   {antes_filtro:,}")
print(f"Registros despues del filtro: {despues_filtro:,}")
print(f"Registros eliminados:         {eliminados:,} ({round(eliminados * 100 / antes_filtro, 2)}%)")

---
## 5. Paso 3: Conversion de Tipos de Datos

El campo `valor_actividad_um` (Activity Value en micromolar) llega como string
desde el archivo JSON y necesita convertirse a tipo double para poder realizar
calculos numericos (promedios, minimos, maximos, etc.).

**Nota tecnica:** Muchos registros tienen este campo vacio (cadena `""`).
Se usa `F.when` para convertir cadenas vacias a NULL antes del casteo,
evitando el error `CAST_INVALID_INPUT` de Spark.

In [ ]:
# Convertir valor_actividad_um de string a double
# Paso 1: Reemplazar cadenas vacias por None (null)
# Paso 2: Castear a DoubleType solo los valores no vacios
# Esto evita el error CAST_INVALID_INPUT que ocurre al intentar castear "" a double
df_tipado = df_filtrado.withColumn(
    "valor_actividad_um",
    F.when(
        (F.col("valor_actividad_um").isNull()) | (F.trim(F.col("valor_actividad_um")) == ""),
        F.lit(None).cast(DoubleType())
    ).otherwise(
        F.col("valor_actividad_um").cast(DoubleType())
    )
)

# Verificar cuantos registros tienen valor numerico valido vs nulo
valores_validos = df_tipado.filter(F.col("valor_actividad_um").isNotNull()).count()
valores_nulos = df_tipado.filter(F.col("valor_actividad_um").isNull()).count()
print(f"Registros con valor de actividad numerico valido: {valores_validos:,}")
print(f"Registros sin valor de actividad (nulo o vacio):  {valores_nulos:,}")

---
## 6. Paso 4: Eliminacion de Duplicados

Se eliminan filas completamente duplicadas para garantizar la unicidad
de los registros. Esto es importante para evitar sesgar los calculos
estadisticos (promedios, conteos) en las consultas analiticas.

In [ ]:
# Contar registros antes de la deduplicacion
antes_dedup = df_tipado.count()

# Eliminar filas completamente duplicadas
# dropDuplicates() sin argumentos compara todas las columnas
df_sin_duplicados = df_tipado.dropDuplicates()

# Contar registros despues de la deduplicacion
despues_dedup = df_sin_duplicados.count()
duplicados_removidos = antes_dedup - despues_dedup

# Reportar impacto de la deduplicacion
print(f"Registros antes de deduplicar:  {antes_dedup:,}")
print(f"Registros despues de deduplicar: {despues_dedup:,}")
print(f"Duplicados eliminados:           {duplicados_removidos:,}")

---
## 7. Paso 5: Enriquecimiento con Propiedades Fisicoquimicas

Se realiza un **LEFT JOIN** entre la tabla de bioactividad y la tabla de
propiedades de compuestos para agregar informacion fisicoquimica relevante:

- **peso_molecular**: Peso molecular del compuesto en Daltons
- **xlogp**: Coeficiente de particion octanol-agua (lipofilicidad)
- **complejidad**: Indice de complejidad molecular
- **donantes_h**: Conteo de donantes de enlaces de hidrogeno
- **aceptores_h**: Conteo de aceptores de enlaces de hidrogeno

Se usa LEFT JOIN para conservar todos los registros de bioactividad,
incluso si alguno no tiene propiedades asociadas.

In [ ]:
# Preparar tabla de propiedades con las columnas necesarias para el JOIN
# Se renombran las columnas de PubChem a nombres en espanol
# Se convierten los tipos de datos numericos
df_props_join = df_compuestos.select(
    F.col("CID").cast("string").alias("_cid_join"),           # Llave de JOIN (temporal)
    F.col("MolecularWeight").cast(DoubleType()).alias("peso_molecular"),  # Peso en Daltons
    F.col("XLogP").cast(DoubleType()).alias("xlogp"),           # Lipofilicidad
    F.col("Complexity").cast(DoubleType()).alias("complejidad"), # Complejidad molecular
    F.col("HBondDonorCount").cast("int").alias("donantes_h"),   # Donantes de H
    F.col("HBondAcceptorCount").cast("int").alias("aceptores_h") # Aceptores de H
)

# Ejecutar LEFT JOIN entre bioactividad y propiedades
# Llave: id_compuesto (Plata) = CID (Propiedades)
df_enriquecido = df_sin_duplicados.join(
    df_props_join,
    df_sin_duplicados["id_compuesto"] == df_props_join["_cid_join"],
    "left"  # LEFT JOIN para conservar todos los registros de bioactividad
).drop("_cid_join")  # Eliminar la columna temporal de JOIN

print(f"Registros despues del enriquecimiento: {df_enriquecido.count():,}")

---
## 8. Paso 6: Clasificacion segun la Regla de los 5 de Lipinski

Se evalua cada compuesto contra los 4 criterios de la Regla de Lipinski
y se agrega una columna de clasificacion binaria:
- `"Cumple Lipinski"`: El compuesto cumple **todos** los criterios
- `"No cumple Lipinski"`: El compuesto viola al menos un criterio

Esta clasificacion es util para el analisis farmacologico en la capa Oro.

In [ ]:
# Aplicar clasificacion de Lipinski evaluando los 4 criterios simultaneamente
df_con_lipinski = df_enriquecido.withColumn(
    "clasificacion_lipinski",
    F.when(
        # Un compuesto cumple Lipinski si satisface TODOS los criterios
        (F.col("peso_molecular") <= 500) &    # Peso molecular <= 500 Da
        (F.col("xlogp") <= 5) &               # XLogP <= 5
        (F.col("donantes_h") <= 5) &           # Donantes de H <= 5
        (F.col("aceptores_h") <= 10),          # Aceptores de H <= 10
        F.lit("Cumple Lipinski")
    ).otherwise(F.lit("No cumple Lipinski"))
)

# Mostrar distribucion de la clasificacion Lipinski
print("Distribucion de clasificacion Lipinski:")
df_con_lipinski.groupBy("clasificacion_lipinski").count().show()

---
## 9. Agregar Trazabilidad y Guardar en Capa Plata

Se agrega la columna de trazabilidad de la capa Plata y se guarda
la tabla final como Delta Table en el esquema de Plata.

In [ ]:
# Agregar columna de trazabilidad con la fecha/hora de la transformacion
df_plata = df_con_lipinski \
    .withColumn("_plata_fecha_transformacion", F.current_timestamp())

# Guardar como Delta Table en la capa Plata
# mode='overwrite' reemplaza la tabla completa en cada ejecucion
df_plata.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{ESQUEMA_PLATA}.actividades_biologicas")

# Verificar la tabla guardada leyendola desde el catalogo
total_plata = spark.table(f"{ESQUEMA_PLATA}.actividades_biologicas").count()
print(f"Tabla creada: {ESQUEMA_PLATA}.actividades_biologicas")
print(f"Registros guardados: {total_plata:,}")

---
## 10. Verificacion Final

Se muestra el esquema y una muestra de datos de la tabla Plata
para validar visualmente que las transformaciones fueron correctas.

In [ ]:
# Mostrar el esquema completo de la tabla Plata
print("--- Esquema de la tabla Plata ---")
spark.table(f"{ESQUEMA_PLATA}.actividades_biologicas").printSchema()

# Mostrar una muestra de los primeros 5 registros
print("--- Muestra de datos ---")
spark.table(f"{ESQUEMA_PLATA}.actividades_biologicas").show(5, truncate=True)

In [ ]:
# Resumen consolidado de la transformacion Bronce a Plata
# Muestra las metricas clave del proceso y la tasa de retencion
print("=" * 60)
print("   RESUMEN DE TRANSFORMACION BRONCE -> PLATA")
print("=" * 60)
print(f"  Registros origen (Bronce):     {total_bronce:,}")
print(f"  Filtrados (sin resultado):     {eliminados:,}")
print(f"  Duplicados removidos:          {duplicados_removidos:,}")
print(f"  Registros destino (Plata):     {total_plata:,}")

# Calcular tasa de retencion: que porcentaje de datos de Bronce llego a Plata
tasa_retencion = round(total_plata * 100 / total_bronce, 1)
print(f"  Tasa de retencion:             {tasa_retencion}%")
print("=" * 60)